# Xdl Download Manager - Colab Demo

**An open-source alternative to Internet Download Manager (IDM)**

Xdl is a powerful, feature-rich download manager that supports video/audio downloads from 1000+ sites, cloud storage services, and generic HTTP/HTTPS file downloads with multi-threaded segmented downloading and resume support.

### Features
- Multi-threaded segmented downloads (up to 32 segments)
- Resume support for paused/interrupted downloads
- 1000+ video/audio sites via yt-dlp (YouTube, Vimeo, TikTok, etc.)
- Cloud storage: Google Drive, MediaFire, Pixeldrain, HuggingFace
- Audio extraction: MP3, AAC, FLAC, Opus, WAV
- Smart category auto-detection
- Gradio web interface for easy interaction

---

## 1. Install Xdl

In [ ]:
# Clone and install dependencies
!git clone https://github.com/BF667/Xdl.git
%cd Xdl
!pip install -q yt-dlp requests beautifulsoup4 tqdm lxml gradio fastapi uvicorn

## 2. Install FFmpeg (required for audio extraction & video merging)

In [ ]:
!apt-get install -y ffmpeg > /dev/null 2>&1
!ffmpeg -version | head -1

## 3. Quick Download - CLI Mode

Use Xdl directly from the command line. Just paste any URL and hit run.

In [ ]:
# Download a file (replace with any URL)
!python3 main.py --cli "https://speed.hetzner.de/100MB.bin" -o /content/downloads

### Download Video from YouTube

In [ ]:
# Download a YouTube video (best quality)
!python3 main.py --cli "https://www.youtube.com/watch?v=dQw4w9WgXcQ" -o /content/downloads

### Download Audio Only (MP3)

In [ ]:
# Download as audio (MP3)
!python3 main.py --cli "https://www.youtube.com/watch?v=dQw4w9WgXcQ" -a -f mp3 -o /content/downloads

### Get URL Info (without downloading)

In [ ]:
# Analyze a URL without downloading
!python3 main.py --info "https://www.youtube.com/watch?v=dQw4w9WgXcQ"

## 4. Launch Gradio Web Interface

Launch a full web-based download manager UI powered by Gradio. A public share link will be generated automatically.

In [ ]:
# Launch Gradio web demo with public share link
!python3 gradio_demo.py --share --no-browser

## 5. Python API - Programmatic Downloads

Use Xdl as a Python library for custom download workflows.

In [ ]:
import sys
sys.path.insert(0, '/content/Xdl')

from xdl.downloaders.router import DownloadRouter
from xdl.core.models import DownloadItem, DownloadStatus, DownloadCategory
from xdl.utils.helpers import format_size, format_speed, format_time
import threading, time

# Initialize the download router
router = DownloadRouter()

# Detect what site a URL belongs to
url = "https://speed.hetzner.de/100MB.bin"
site = router.detect_site(url)
print(f"Detected site: {site}")

# Get URL info without downloading
info = router.get_info(url)
print(f"Filename: {info.get('filename', 'Unknown')}")
print(f"File size: {format_size(info.get('file_size', 0))}")

# Create a download item
item = router.create_item(url=url, save_path="/content/downloads")
print(f"\nStarting download: {item.filename}")
print(f"Save path: {item.save_path}")
print(f"Category: {item.category.value}")

In [ ]:
# Run download with progress tracking
stop_event = threading.Event()
last_time = [time.time()]

def on_progress(dl_item):
    now = time.time()
    if now - last_time[0] >= 1.0:
        last_time[0] = now
        bar_len = 40
        filled = int(bar_len * dl_item.progress / 100)
        bar = '█' * filled + '░' * (bar_len - filled)
        print(
            f"\r  [{bar}] {dl_item.progress:.1f}% | "
            f"{format_speed(dl_item.speed)} | "
            f"ETA: {format_time(dl_item.eta)}",
            end='', flush=True
        )

downloader = router.get_downloader(url)
downloader.on_progress = on_progress

print(f"Downloading: {item.filename}")
thread = downloader.start(item)
thread.join()

if item.status == DownloadStatus.COMPLETED:
    print(f"\n\nDownload complete: {item.save_path}/{item.filename}")
else:
    print(f"\n\nStatus: {item.status.value}")

## 6. Download from Cloud Storage

In [ ]:
# Google Drive (replace FILE_ID with actual file ID)
# !python3 main.py --cli "https://drive.google.com/file/d/FILE_ID/view" -o /content/downloads

# MediaFire
# !python3 main.py --cli "https://www.mediafire.com/file/xxxxx/filename.zip/file" -o /content/downloads

# Pixeldrain
# !python3 main.py --cli "https://pixeldrain.com/u/FILE_ID" -o /content/downloads

# HuggingFace models
# !python3 main.py --cli "https://huggingface.co/model-name/resolve/main/model.bin" -o /content/downloads

# Any direct HTTP/HTTPS link
!python3 main.py --cli "https://speed.hetzner.de/100MB.bin" -o /content/downloads

## 7. Browse Downloaded Files

In [ ]:
import os

download_dir = "/content/downloads"
if os.path.exists(download_dir):
    files = os.listdir(download_dir)
    print(f"Files in {download_dir}: ({len(files)} items)\n")
    for f in sorted(files):
        fpath = os.path.join(download_dir, f)
        size = os.path.getsize(fpath) if os.path.isfile(fpath) else 0
        print(f"  {f}  ({format_size(size)})")
else:
    print("No downloads yet. Run the cells above to start downloading!")

---

### Links
- **GitHub**: [https://github.com/BF667/Xdl](https://github.com/BF667/Xdl)
- **License**: Unlicense (Public Domain)
- **Powered by**: [yt-dlp](https://github.com/yt-dlp/yt-dlp), [Gradio](https://gradio.app), [FastAPI](https://fastapi.tiangolo.com)